# 04 - Pengujian akhir

Bandingkan Whisper Base asli, Model A, dan Model B pada FLEURS test Indonesia dan Inggris. Gunakan checkpoint yang dipilih berdasarkan dev.

Test Model A dan Model B sudah pernah dibuka pada eksperimen sebelumnya. Hasil ini dicatat sebagai perbandingan pada test yang sudah dilihat.


## Pengaturan

Siapkan pustaka, data test lokal, serta checkpoint.


In [ ]:
%pip -q install 'transformers==4.46.3' 'peft==0.13.2' 'accelerate==1.1.1' 'safetensors>=0.4.5'


In [ ]:
from pathlib import Path
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
import csv, json, math, time, unicodedata
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from IPython.display import display, Markdown, HTML
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = root / "data"
CHECKPOINT_A = root / "checkpoints" / "model_a" / "best.pt"
CHECKPOINT_B = root / "checkpoints" / "model_b" / "whisper_base_lora_v1" / "best.pt"
CACHE_ROOT = root / "data" / "preprocessed" / "model_b"
MODEL_ID = "openai/whisper-base"
LANGUAGES = ("id_id", "en_us")
LANGUAGE_NAMES = {"id_id": "Indonesia", "en_us": "Inggris"}
WHISPER_LANG = {"id_id": "indonesian", "en_us": "english"}
SAMPLE_RATE, N_FFT, HOP, N_MELS = 16000, 400, 160, 80

for checkpoint in (CHECKPOINT_A, CHECKPOINT_B):
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {checkpoint}")
print("Perangkat:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Checkpoint Model A:", CHECKPOINT_A)
print("Checkpoint Model B:", CHECKPOINT_B)


## Data test dan metrik

Baca audio dan transkrip dengan normalisasi yang sama untuk ketiga model.


In [ ]:
duration_samples = {}

def normalize_text(text):
    text = unicodedata.normalize("NFC", text).casefold().replace("’", "'")
    return " ".join("".join(c if c.isalnum() or c == "'" else " " for c in text).split())


def read_split(split):
    rows = []
    for language in LANGUAGES:
        path = DATA_ROOT / language / f"{split}.tsv"
        audio_dir = DATA_ROOT / language / "audio" / split
        print(f"Memeriksa {LANGUAGE_NAMES[language]} / {split}...", flush=True)
        available = {entry.name for entry in audio_dir.glob("*.wav")}
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for number, fields in enumerate(csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE), 1):
                if len(fields) != 7 or not fields[0].isdigit():
                    raise ValueError(f"TSV tidak valid: {path}, baris {number}")
                filename = fields[1]
                if Path(filename).name != filename or not filename.lower().endswith(".wav"):
                    raise ValueError(f"Nama WAV tidak valid: {path}, baris {number}")
                audio = audio_dir / filename
                if filename not in available:
                    raise FileNotFoundError(audio)
                if not fields[5].isdigit() or not normalize_text(fields[3]):
                    raise ValueError(f"Durasi atau transkrip tidak valid: {path}, baris {number}")
                duration_samples[audio] = int(fields[5])
                rows.append((language, audio, normalize_text(fields[3]), fields[0]))
    keys = [(language, path.name) for language, path, _, _ in rows]
    if len(set(keys)) != len(keys):
        raise ValueError(f"Ada nama WAV ganda di {split}. Tinjau data.")
    return rows


def read_audio(path):
    rate, pcm = wavfile.read(path)
    samples = pcm.astype(np.float32)
    if np.issubdtype(pcm.dtype, np.integer):
        info = np.iinfo(pcm.dtype)
        if info.min == 0:
            midpoint = (info.max + 1) / 2
            samples = (samples - midpoint) / midpoint
        else:
            samples /= -info.min
    if samples.ndim == 2:
        samples = samples.mean(axis=1)
    if rate <= 0 or not samples.size or not np.isfinite(samples).all():
        raise ValueError(f"Audio tidak valid: {path}")
    if rate != 16000:
        divisor = math.gcd(rate, 16000)
        samples = resample_poly(samples, 16000 // divisor, rate // divisor).astype(np.float32)
    return samples


def feature_for(path):
    relative = path.relative_to(DATA_ROOT)
    signature = f"{path.stat().st_size}-{path.stat().st_mtime_ns}"
    destination = CACHE_ROOT / relative.parent / f"{path.stem}-{signature}.npz"
    if destination.is_file():
        with np.load(destination, allow_pickle=False) as cached:
            return cached["features"], cached["attention_mask"]
    samples = read_audio(path)
    long_audio = len(samples) > 30 * 16000
    encoded = processor.feature_extractor(
        samples, sampling_rate=16000, return_tensors="np", return_attention_mask=True,
        truncation=not long_audio, padding="longest" if long_audio else "max_length"
    )
    features, attention_mask = encoded.input_features[0], encoded.attention_mask[0]
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".tmp")
    with temporary.open("wb") as stream:
        np.savez(stream, features=features, attention_mask=attention_mask)
    temporary.replace(destination)
    return features, attention_mask

test_rows = read_split("test")
print(f"Data test: {len(test_rows):,} sampel")


## Fungsi evaluasi

Siapkan preprocessing, prediksi, dan perhitungan WER serta CER.


In [ ]:
def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, expected in enumerate(reference, 1):
        current = [i]
        for j, actual in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (expected != actual)))
        previous = current
    return previous[-1]


def prefetch_features(rows, workers=4, prefetch=8):
    source = iter(rows)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = deque()
        for _ in range(min(prefetch, len(rows))):
            row = next(source)
            pending.append((row, pool.submit(feature_for, row[1])))
        while pending:
            row, future = pending.popleft()
            features = future.result()
            next_row = next(source, None)
            if next_row is not None:
                pending.append((next_row, pool.submit(feature_for, next_row[1])))
            yield row, features


def progress_batches(items, title, total=None):
    if total is None:
        total = len(items)
    done = 0
    started = time.perf_counter()
    def render(status):
        percent = done * 100 / total if total else 100
        return HTML(f"<b>{title}</b> — {status} · {done}/{total} · {percent:.0f}% · {time.perf_counter()-started:.0f} dtk"
                    f"<div style='height:8px;background:#ddd'><div style='height:100%;width:{percent:.1f}%;background:#008b83'></div></div>")
    view = display(render("berjalan"), display_id=True)
    try:
        for item in items:
            yield item
            done += 1
            if done % 10 == 0 or done == total:
                view.update(render("berjalan"))
    finally:
        view.update(render("selesai" if done == total else "terhenti"))


@torch.inference_mode()
def evaluate(model, rows, title):
    model.eval()
    counts = defaultdict(lambda: [0, 0, 0, 0])
    examples = {}
    batches = prefetch_features(rows)
    for (language, path, reference, _), (features, attention_mask) in progress_batches(
        batches, title, total=len(rows)
    ):
        features = torch.from_numpy(features).unsqueeze(0).to(DEVICE)
        attention_mask = torch.from_numpy(attention_mask).unsqueeze(0).to(DEVICE)
        ids = model.generate(
            input_features=features, attention_mask=attention_mask,
            language=WHISPER_LANG[language], task="transcribe", max_new_tokens=440,
            return_timestamps=features.shape[-1] > 3000
        )
        prediction = normalize_text(processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])
        words = reference.split()
        word_errors = edit_distance(words, prediction.split())
        counts[language][0] += word_errors
        counts[language][1] += len(words)
        counts[language][2] += edit_distance(reference, prediction)
        counts[language][3] += len(reference)
        if words:
            sample = {"reference": reference, "prediction": prediction,
                      "wer": word_errors / len(words), "filename": path.name}
            if language not in examples:
                examples[language] = {"best": sample, "worst": sample}
            if sample["wer"] < examples[language]["best"]["wer"]:
                examples[language]["best"] = sample
            if sample["wer"] > examples[language]["worst"]["wer"]:
                examples[language]["worst"] = sample
    result = {lang: {"wer": w/n, "cer": c/m, "reference_words": n}
              for lang, (w,n,c,m) in counts.items()}
    return {"per_language": result, "macro_wer": sum(result[l]["wer"] for l in LANGUAGES)/len(LANGUAGES)}, examples


def show_results(title, metrics, examples):
    lines = [f"### {title}", "", "| Bahasa | WER | CER |", "| --- | ---: | ---: |"]
    for lang in LANGUAGES:
        row = metrics["per_language"][lang]
        lines.append(f"| {LANGUAGE_NAMES[lang]} | {row['wer']:.1%} | {row['cer']:.1%} |")
    display(Markdown("\n".join(lines)))
    for lang in LANGUAGES:
        display(Markdown(f"**{LANGUAGE_NAMES[lang]}**"))
        for key, title in (("best", "Terbaik"), ("worst", "Terburuk")):
            sample = examples[lang][key]
            print(f"{title} · WER {sample['wer']:.1%} · {sample['filename']}")
            print(f"Acuan: {sample['reference']}")
            prediction = sample["prediction"] or "(kosong)"
            if len(prediction) > 600:
                prediction = prediction[:600] + " ... [dipersingkat untuk tampilan]"
            print(f"Prediksi: {prediction}\n")


def make_mel_bank():
    def hz_to_mel(hz):
        return 2595 * math.log10(1 + hz / 700)

    mel_points = torch.linspace(hz_to_mel(0), hz_to_mel(SAMPLE_RATE / 2), N_MELS + 2)
    left, center, right = (700 * (10 ** (mel_points / 2595) - 1)).unfold(0, 3, 1).T
    fft_hz = torch.linspace(0, SAMPLE_RATE / 2, N_FFT // 2 + 1)
    rising = (fft_hz - left[:, None]) / (center - left)[:, None]
    falling = (right[:, None] - fft_hz) / (right - center)[:, None]
    return torch.minimum(rising, falling).clamp_min(0)


def audio_features(path):
    if not path.is_file():
        raise FileNotFoundError(f"{path} belum ada. Ekstrak arsip WAV sebelum menjalankan notebook.")
    rate, pcm = wavfile.read(path)
    samples = pcm.astype(np.float32)
    if np.issubdtype(pcm.dtype, np.integer):
        info = np.iinfo(pcm.dtype)
        if info.min == 0:
            midpoint = (info.max + 1) / 2
            samples = (samples - midpoint) / midpoint
        else:
            samples /= -info.min
    if samples.ndim == 2:
        samples = samples.mean(axis=1)
    if rate <= 0 or samples.size == 0 or not np.isfinite(samples).all():
        raise ValueError(f"Audio kosong atau tidak valid: {path}")
    if rate != SAMPLE_RATE:
        divisor = math.gcd(rate, SAMPLE_RATE)
        samples = resample_poly(samples, SAMPLE_RATE // divisor, rate // divisor)
    signal = torch.from_numpy(samples)
    power = torch.stft(signal, N_FFT, HOP, window=WINDOW, pad_mode="constant",
                       return_complex=True).abs().square()
    mel = (MEL_BANK @ power).clamp_min(1e-10).log()
    return mel.T.contiguous()


class ModelA(nn.Module):
    def __init__(self, vocab_size, hidden, layers, channels):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(N_MELS, channels[0], kernel_size=5, stride=2, padding=2),
            nn.Conv1d(channels[0], channels[1], kernel_size=3, stride=1, padding=1),
        ])
        self.gru = nn.GRU(channels[1], hidden, num_layers=layers,
                          batch_first=True, bidirectional=True,
                          dropout=0.1 if layers > 1 else 0)
        self.projection = nn.Linear(hidden * 2, vocab_size)

    def forward(self, features, lengths):
        features = features.transpose(1, 2)
        for conv in self.convs:
            features = torch.relu(conv(features))
            lengths = (lengths + 2 * conv.padding[0] - conv.kernel_size[0]) // conv.stride[0] + 1
            valid = torch.arange(features.size(-1), device=features.device) < lengths.to(features.device)[:, None]
            features = features.masked_fill(~valid[:, None, :], 0)
        packed = pack_padded_sequence(features.transpose(1, 2), lengths,
                                      batch_first=True, enforce_sorted=False)
        features, _ = pad_packed_sequence(self.gru(packed)[0], batch_first=True)
        return self.projection(features).log_softmax(-1), lengths


def decode(token_ids, vocab):
    chars = []
    previous = None
    for token in token_ids:
        if token != 0 and token != previous:
            chars.append(vocab[token])
        previous = token
    return "".join(chars)


MEL_BANK = make_mel_bank()
WINDOW = torch.hann_window(N_FFT)

@torch.inference_mode()
def evaluate_a(model, vocab, rows):
    model.eval()
    counts = defaultdict(lambda: [0, 0, 0, 0])
    examples = {}
    batch_size = 16
    starts = range(0, len(rows), batch_size)
    for start in progress_batches(starts, "Model A · test"):
        batch = rows[start:start + batch_size]
        features = [audio_features(path) for _, path, _, _ in batch]
        lengths = torch.tensor([len(feature) for feature in features], dtype=torch.long)
        padded = pad_sequence(features, batch_first=True)
        predictions, output_lengths = model(padded.to(DEVICE), lengths)
        ids = predictions.argmax(-1).cpu().tolist()
        for (language, path, reference, _), tokens, length in zip(batch, ids, output_lengths.tolist()):
            prediction = normalize_text(decode(tokens[:length], vocab))
            words = reference.split()
            if not words or not reference:
                raise ValueError(f"Transkrip kosong: {path}")
            errors = edit_distance(words, prediction.split())
            counts[language][0] += errors
            counts[language][1] += len(words)
            counts[language][2] += edit_distance(reference, prediction)
            counts[language][3] += len(reference)
            sample = {"reference": reference, "prediction": prediction,
                      "wer": errors / len(words), "filename": path.name}
            if language not in examples:
                examples[language] = {"best": sample, "worst": sample}
            if sample["wer"] < examples[language]["best"]["wer"]:
                examples[language]["best"] = sample
            if sample["wer"] > examples[language]["worst"]["wer"]:
                examples[language]["worst"] = sample
    results = {lang: {"wer": w/n, "cer": c/m, "reference_words": n}
               for lang, (w,n,c,m) in counts.items()}
    return {"per_language": results,
            "macro_wer": sum(results[lang]["wer"] for lang in LANGUAGES) / len(LANGUAGES)}, examples


## Whisper Base asli

Ukur Whisper sebelum LoRA pada data test.


In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID)
whisper = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
whisper.generation_config.forced_decoder_ids = None
baseline_metrics, baseline_examples = evaluate(whisper, test_rows, "Whisper asli · test")
show_results("Whisper asli · test", baseline_metrics, baseline_examples)


## Model A

Ukur checkpoint CNN + BiGRU + CTC terbaik berdasarkan dev.


In [ ]:
checkpoint_a = torch.load(CHECKPOINT_A, map_location=DEVICE, weights_only=True)
vocab_a = checkpoint_a["settings"]["vocab"]
model_a = ModelA(len(vocab_a), **checkpoint_a["settings"]["model"]).to(DEVICE)
model_a.load_state_dict(checkpoint_a["model"])
metrics_a, examples_a = evaluate_a(model_a, vocab_a, test_rows)
show_results(f"Model A · epoch {checkpoint_a['epoch']} · test", metrics_a, examples_a)
del model_a
torch.cuda.empty_cache()


## Model B

Muat adapter LoRA terbaik berdasarkan dev dan ukur pada data test.


In [ ]:
checkpoint_b = torch.load(CHECKPOINT_B, map_location=DEVICE, weights_only=True)
settings_b = checkpoint_b["settings"]
if settings_b["model"] != MODEL_ID:
    raise ValueError("Model dasar checkpoint LoRA tidak sesuai.")
lora = LoraConfig(r=settings_b["rank"], lora_alpha=settings_b["alpha"],
                  lora_dropout=settings_b["dropout"],
                  target_modules=settings_b["targets"], bias="none")
model_b = get_peft_model(whisper, lora)
set_peft_model_state_dict(model_b, checkpoint_b["adapter"])
metrics_b, examples_b = evaluate(model_b, test_rows, "Model B · test")
show_results(f"Model B · epoch {checkpoint_b['epoch']} · test", metrics_b, examples_b)


## Perbandingan

Bandingkan WER dan CER tiap bahasa. Nilai lebih rendah berarti error lebih sedikit.


In [ ]:
import matplotlib.pyplot as plt

results = [("Whisper asli", baseline_metrics),
           (f"Model A · epoch {checkpoint_a['epoch']}", metrics_a),
           (f"Model B · epoch {checkpoint_b['epoch']}", metrics_b)]
lines = ["| Model | Bahasa | WER | CER |", "| --- | --- | ---: | ---: |"]
for name, metrics in results:
    for language in LANGUAGES:
        row = metrics["per_language"][language]
        lines.append(f"| {name} | {LANGUAGE_NAMES[language]} | {row['wer']:.1%} | {row['cer']:.1%} |")
display(Markdown("\n".join(lines)))

labels = ["Indonesia\nWER", "Indonesia\nCER", "Inggris\nWER", "Inggris\nCER"]
positions = np.arange(len(labels))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
all_values = []
for offset, ((name, metrics), color) in enumerate(zip(results, ("#007C83", "#6D54AD", "#D97706"))):
    values = [metrics["per_language"][language][metric] * 100
              for language in LANGUAGES for metric in ("wer", "cer")]
    all_values.extend(values)
    bars = ax.bar(positions + (offset - 1) * width, values, width, label=name, color=color)
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_xticks(positions, labels)
ax.set_ylabel("Error (%)")
ax.set_title("Perbandingan pada data test")
ax.set_ylim(0, max(all_values + [1]) * 1.2)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
ax.legend()
plt.show()
